# REVELA DBSCAN: Parameter Sensitivity (273-Record Snapshot)

This separate sensitivity notebook uses the available 273-row CSV (`QUERY RESULT.csv`). It compares four configurations: 50 m / MinPts 3, 150 m / MinPts 3, 150 m / MinPts 5, and 300 m / MinPts 3. It does not update application settings or database records.

The snapshot contains 272 Red records and 1 Black record, all from the detection source. DBSCAN labels individual geospatial records, not unique businesses. Cluster counts, noise percentage, and silhouette are descriptive outputs; they do not establish an optimal setting, prove a real hotspot, or measure detection accuracy.

## Input and analysis limits

The input is the available eligible-only CSV with columns `record_key`, `latitude`, `longitude`, `effective_color`, `source`, `registry_linked`, `barangayID`, and `detected_date`. Coordinates are required and effective flags must be Red or Black. No names, addresses, business IDs, or database connection are used. The four settings are evaluated on the same ordered records using Haversine distance; `min_samples` includes the point itself. Silhouette is computed for non-noise records only when at least two clusters exist; it is not an accuracy metric or a parameter-selection objective.

In [1]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import platform

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn
from sklearn.cluster import DBSCAN
from sklearn.metrics import adjusted_rand_score, silhouette_score
from sklearn.metrics.pairwise import haversine_distances
from sklearn.neighbors import NearestNeighbors

pd.set_option("display.max_columns", 30)
pd.set_option("display.max_colwidth", 100)
print(f"Python {platform.python_version()} | pandas {pd.__version__} | NumPy {np.__version__} | scikit-learn {sklearn.__version__}")

Python 3.13.3 | pandas 3.0.3 | NumPy 2.5.1 | scikit-learn 1.9.0


The next cell reads the available `QUERY RESULT.csv` from the Downloads folder. No database connection is used.

In [2]:
EXPORT_DIR = Path(r"C:\Users\Mateo\Downloads")
INPUT_PATH = EXPORT_DIR / "QUERY RESULT.csv"
METADATA_PATH = EXPORT_DIR / "QUERY RESULT.metadata.json"
EXPECTED_COLUMNS = {
    "record_key", "latitude", "longitude", "effective_color", "source",
    "registry_linked", "barangayID", "detected_date",
}

if not INPUT_PATH.is_file():
    raise FileNotFoundError(f"Expected manual export was not found: {INPUT_PATH}")

csv_sha256 = hashlib.sha256(INPUT_PATH.read_bytes()).hexdigest()
raw_df = pd.read_csv(
    INPUT_PATH,
    dtype={"record_key": "string", "effective_color": "string", "source": "string"},
    keep_default_na=False,
)
missing_columns = sorted(EXPECTED_COLUMNS.difference(raw_df.columns))
unexpected_columns = sorted(set(raw_df.columns).difference(EXPECTED_COLUMNS))
if missing_columns or unexpected_columns:
    raise ValueError(f"CSV schema mismatch; missing={missing_columns}, unexpected={unexpected_columns}")
if raw_df.empty:
    raise ValueError("The supplied CSV is empty; no data was fabricated or substituted.")

raw_df["record_key"] = raw_df["record_key"].str.strip()
raw_df["effective_color"] = raw_df["effective_color"].str.strip()
raw_df["latitude"] = pd.to_numeric(raw_df["latitude"], errors="coerce")
raw_df["longitude"] = pd.to_numeric(raw_df["longitude"], errors="coerce")
missing_keys = int(raw_df["record_key"].eq("").sum())
duplicate_keys = int(raw_df["record_key"].duplicated().sum())
if missing_keys or duplicate_keys:
    raise ValueError(f"Surrogate keys must be present and unique; missing={missing_keys}, duplicate={duplicate_keys}.")
if raw_df[["latitude", "longitude"]].isna().any().any():
    raise ValueError("CSV has missing or non-numeric coordinates; investigate the export rather than silently dropping rows.")
if not raw_df["latitude"].between(-90, 90).all() or not raw_df["longitude"].between(-180, 180).all():
    raise ValueError("Coordinates must be decimal degrees within valid latitude/longitude ranges.")
if not raw_df["effective_color"].str.casefold().isin({"red", "black"}).all():
    raise ValueError("Objective export must contain only effective Red or Black records.")

raw_df["effective_color"] = raw_df["effective_color"].str.title()
raw_df["registry_linked"] = raw_df["registry_linked"].astype("string").str.casefold().map(
    {"1": True, "true": True, "yes": True, "0": False, "false": False, "no": False}
)
if raw_df["registry_linked"].isna().any():
    raise ValueError("registry_linked must be a recognizable boolean value for every row.")
eligible_df = raw_df.copy()

print(f"CSV: {INPUT_PATH.name} | rows={len(eligible_df):,} | SHA-256={csv_sha256}")
print("Columns validated; no unexpected fields (including names, addresses, place IDs, reporter IDs, or business IDs).")
print("Rows by effective color:", eligible_df["effective_color"].value_counts().to_dict())
print("Rows by source:", eligible_df["source"].replace("", "(blank)").value_counts().to_dict())
print(f"Registry-linked rows: {int(eligible_df['registry_linked'].sum()):,}")
print(f"Duplicate rounded coordinate rows: {int(eligible_df.duplicated(['latitude', 'longitude']).sum()):,}")
print(f"Metadata sidecar present: {METADATA_PATH.is_file()}")


CSV: QUERY RESULT.csv | rows=273 | SHA-256=1597ab61ef59ba9516275ca4dd0b0c80beab29f7046b801262e279266d3074ea
Columns validated; no unexpected fields (including names, addresses, place IDs, reporter IDs, or business IDs).
Rows by effective color: {'Red': 272, 'Black': 1}
Rows by source: {'detection': 273}
Registry-linked rows: 0
Duplicate rounded coordinate rows: 5
Metadata sidecar present: True


## Prepare the ordered coordinates

The key must be unique and sequential, preserving the original export order. The analysis below will stop rather than silently dropping invalid coordinates or rows.

In [3]:
EARTH_RADIUS_METERS = 6_371_008.8

record_order = pd.to_numeric(eligible_df["record_key"], errors="coerce")
if record_order.isna().any() or record_order.duplicated().any():
    raise ValueError("record_key must be a unique sequential numeric surrogate.")
if set(record_order.astype(int)) != set(range(1, len(eligible_df) + 1)):
    raise ValueError("record_key must be sequential 1..N to preserve export order.")

eligible_df = eligible_df.assign(_stable_order=record_order).sort_values(
    "_stable_order", kind="stable"
).reset_index(drop=True)

if len(eligible_df) != 273:
    raise ValueError(f"Expected the available 273-record snapshot; found {len(eligible_df)} rows.")

coords_radians = np.radians(eligible_df[["latitude", "longitude"]].to_numpy(dtype=float))
distance_matrix_radians = haversine_distances(coords_radians)
print(f"Validated analysis snapshot: {len(eligible_df)} records")


Validated analysis snapshot: 273 records


## Run the four configurations

The 50 m / MinPts 3 setting is included as the repository-default reference, not as a verified live setting. The two 150 m rows isolate the MinPts change; the 50 m and 300 m rows at MinPts 3 show the radius comparison.

In [4]:
SCENARIOS = [(50, 3), (150, 3), (150, 5), (300, 3)]


def evaluate_dbscan(eps_meters, min_samples):
    labels = DBSCAN(
        eps=float(eps_meters) / EARTH_RADIUS_METERS,
        min_samples=int(min_samples),
        algorithm="ball_tree",
        metric="haversine",
    ).fit_predict(coords_radians)
    clustered_mask = labels != -1
    clustered_labels = labels[clustered_mask]
    cluster_sizes = pd.Series(clustered_labels).value_counts().sort_values(ascending=False)
    cluster_count = int(cluster_sizes.size)
    clustered_count = int(clustered_mask.sum())
    silhouette = None
    if cluster_count >= 2 and clustered_count > cluster_count:
        clustered_distances = distance_matrix_radians[np.ix_(clustered_mask, clustered_mask)]
        silhouette = float(silhouette_score(
            clustered_distances, clustered_labels, metric="precomputed"
        ))
    return {
        "eps_m": float(eps_meters),
        "min_samples": int(min_samples),
        "records": len(labels),
        "cluster_count": cluster_count,
        "clustered_records": clustered_count,
        "noise_records": int((~clustered_mask).sum()),
        "noise_percent": 100 * float((~clustered_mask).sum()) / len(labels),
        "silhouette_nonnoise_only": silhouette,
        "largest_cluster_size": int(cluster_sizes.iloc[0]) if cluster_count else 0,
        "cluster_sizes_desc": [int(size) for size in cluster_sizes.tolist()],
        "labels": labels,
    }

scenario_results = {
    scenario: evaluate_dbscan(*scenario)
    for scenario in SCENARIOS
}
results_df = pd.DataFrame([
    {key: value for key, value in result.items() if key != "labels"}
    for result in scenario_results.values()
])
display(results_df.round({"noise_percent": 1, "silhouette_nonnoise_only": 3}))
print("Noise is DBSCAN label -1. Silhouette excludes noise; it does not establish accuracy or an optimal configuration.")

,eps_m,min_samples,records,cluster_count,clustered_records,noise_records,noise_percent,silhouette_nonnoise_only,largest_cluster_size,cluster_sizes_desc
0,50.0,3,273,17,74,199,72.9,0.709,14,"[14, 10, 5, 4, 4, 4, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3]"
1,150.0,3,273,29,222,51,18.7,0.410,70,"[70, 39, 8, 7, 6, 6, 6, 5, 5, 5, 5, 4, 4, 4, 4, 4, 4, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3]"
2,150.0,5,273,12,151,122,44.7,0.509,55,"[55, 38, 8, 8, 6, 6, 5, 5, 5, 5, 5, 5]"
3,300.0,3,273,17,266,7,2.6,0.331,149,"[149, 16, 15, 12, 11, 9, 8, 6, 6, 6, 6, 5, 4, 4, 3, 3, 3]"


Noise is DBSCAN label -1. Silhouette excludes noise; it does not establish accuracy or an optimal configuration.


## Interpretation

Use the actual table above to populate the thesis table. Do not retain previously drafted cluster/noise values unless they match this run. Describe the output as parameter sensitivity on this specific 199-record snapshot. Noise means a record did not belong to a density-connected cluster at that setting; it does not imply the record should be excluded from inspection. DBSCAN is not constrained by barangay boundaries, so claims about cross-barangay merging require checking cluster membership and locations separately.